# EchoAI — Qwen2.5-3B-Instruct fine-tune · Step 2: Tier B synthesis + final dataset

Runs on a free Colab T4. ~1.5 hours.

**Inputs (upload to Colab or pull from Drive):**
- `constitution_ir.json`
- `dataset_tier_a.jsonl` (from Step 1)
- `val_prompts.jsonl`, `heldout_prompts.jsonl`

**Outputs (written to Drive):**
- `dataset_v1_train.jsonl` (~15-20k samples, ChatML)
- `dataset_v1_val.jsonl` (~1-2k samples)
- `heldout_prompts.jsonl` (copy, untouched)

## Cell 1 — Setup (runtime + Drive)

Before running: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch, sys, subprocess
assert torch.cuda.is_available(), 'No GPU! Switch runtime to T4.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

# Mount Drive for inputs/outputs
from google.colab import drive
drive.mount('/content/drive')

# Workspace
WORK = '/content/qwen_ft'
DRIVE_DIR = '/content/drive/MyDrive/echoai_qwen_ft'   # change if you want a different folder
subprocess.run(['mkdir', '-p', WORK, DRIVE_DIR], check=True)
print('Workspace :', WORK)
print('Drive dir :', DRIVE_DIR)

In [ ]:
# Install deps (T4-compatible versions)
!pip install -q -U transformers==4.46.3 accelerate==1.1.1 bitsandbytes==0.44.1 peft==0.13.2 datasets==3.1.0 trl==0.12.1
print('\nInstalled. Restart NOT required.')

## Cell 2 — Upload inputs

Either upload from local OR copy from Drive. Pick one.

**Option A (upload from local):** run the cell below, click "Choose Files" and select the 4 files:
`constitution_ir.json`, `dataset_tier_a.jsonl`, `val_prompts.jsonl`, `heldout_prompts.jsonl`.

In [ ]:
# OPTION A — upload from local
from google.colab import files
import shutil, os
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'{WORK}/{name}')
    print(f'  staged: {WORK}/{name}')

# Also copy to Drive so we don't re-upload
for f in ['constitution_ir.json','dataset_tier_a.jsonl','val_prompts.jsonl','heldout_prompts.jsonl']:
    if os.path.exists(f'{WORK}/{f}'):
        shutil.copy(f'{WORK}/{f}', f'{DRIVE_DIR}/{f}')
print('Backed up to Drive.')

In [ ]:
# OPTION B — copy from Drive (use this on session re-launch / account switch)
import shutil
for f in ['constitution_ir.json','dataset_tier_a.jsonl','val_prompts.jsonl','heldout_prompts.jsonl']:
    src = f'{DRIVE_DIR}/{f}'
    dst = f'{WORK}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst); print(f'  copied: {f}')
    else:
        print(f'  MISSING in Drive: {f}')

## Cell 3 — Load Qwen2.5-1.5B-Instruct (synthesis model, 4-bit)

Why 1.5B not 3B for synthesis? 2× faster on T4 with adequate quality for Q/A generation. We'll load the 3B as the *training* target in the next notebook.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

SYNTH_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(SYNTH_MODEL)
model = AutoModelForCausalLM.from_pretrained(SYNTH_MODEL, quantization_config=bnb, device_map='auto')
model.eval()
print('Synth model loaded:', SYNTH_MODEL)
print('Mem (GB):', round(torch.cuda.memory_allocated()/1e9, 2))

## Cell 4 — Tier B synthesis: 3 Q/A per sampled article

We iterate over ~500 articles (sampled across all parts), prompt the synthesis model for 3 Q/A pairs, verify the answer cites the right Article number, and save the cleaned samples.

In [ ]:
import json, re, random, time
from pathlib import Path

random.seed(42)
ir = json.load(open(f'{WORK}/constitution_ir.json'))

def all_articles(ir):
    out = []
    for part in ir['parts']:
        for a in part.get('articles', []) or []:
            out.append((part, None, a))
        for ch in part.get('chapters', []) or []:
            for a in ch.get('articles', []) or []:
                out.append((part, ch, a))
    return out

ARTICLES = [(p, c, a) for (p, c, a) in all_articles(ir) if a.get('number')]
print(f'Total articles: {len(ARTICLES)}')

# Sample 500 articles (or all if fewer)
N_ARTICLES = min(500, len(ARTICLES))
SAMPLED = random.sample(ARTICLES, N_ARTICLES)
print(f'Will synthesize Q/A for {N_ARTICLES} articles.')

In [ ]:
SYNTH_SYSTEM = (
    'You generate training data. Given an article of the Constitution of Pakistan, produce exactly 3 question/answer pairs as JSON. '
    'Each answer MUST quote the source verbatim and end with the citation in square brackets like [Art 25(1)].\n\n'
    'Vary the question style: 1 direct lookup, 1 paraphrased ("how does the Constitution treat X"), 1 definition/explanation. '
    'Output ONLY a JSON array of 3 objects with keys q, a. No prose, no markdown.'
)

def article_text(a):
    parts = []
    if a.get('text'): parts.append(a['text'].strip())
    for cl in a.get('clauses', []) or []:
        cid = cl.get('id','')
        ctext = (cl.get('text') or '').strip()
        if ctext: parts.append(f'{cid} {ctext}')
        for sc in cl.get('subclauses', []) or []:
            sid = sc.get('id',''); stxt = (sc.get('text') or '').strip()
            if stxt: parts.append(f'    {sid} {stxt}')
    return '\n'.join(parts).strip()

def build_prompt(part, chapter, article):
    loc = f'Part {part.get("number","")}' + (f', Chapter {chapter["number"]}' if chapter else '')
    mn = article.get('marginal_note') or ''
    body = article_text(article)
    user = f'Article {article["number"]} ({loc}) — {mn}\n\n{body}\n\nGenerate 3 Q/A pairs as a JSON array.'
    msgs = [
        {'role':'system', 'content': SYNTH_SYSTEM},
        {'role':'user', 'content': user},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=512):
    inputs = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=2048).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
    return [tok.decode(out[i, inputs.input_ids.shape[1]:], skip_special_tokens=True) for i in range(len(prompts))]

# Quick smoke test (1 article)
p, c, a = SAMPLED[0]
print('Smoke test on Article', a['number'])
print(generate_batch([build_prompt(p, c, a)])[0][:600])

In [ ]:
# Full batched synthesis loop
BATCH = 4   # T4 can usually handle 4 concurrent prompts at 2k ctx
out_path = Path(f'{WORK}/dataset_tier_b_raw.jsonl')
out_path.write_text('')   # truncate

SYSTEM_PROMPT = (
    'You are an expert on the Constitution of the Islamic Republic of Pakistan. '
    'Answer using only the text of the Constitution. Quote the relevant clause verbatim '
    'and end every answer with the citation in square brackets, e.g. [Art 25(1)] or '
    "[Art 260(1)]. If the question is not addressed in the Constitution, reply with: "
    "'This is not addressed in the Constitution of Pakistan. [no citation]'"
)

CITE_RE = re.compile(r'\[Art\s*([\d]+[A-Z]?)')
kept = bad = 0
t0 = time.time()
for i in range(0, len(SAMPLED), BATCH):
    chunk = SAMPLED[i:i+BATCH]
    prompts = [build_prompt(p, c, a) for (p, c, a) in chunk]
    raws = generate_batch(prompts)
    for (p, c, a), raw in zip(chunk, raws):
        # extract JSON array
        m = re.search(r'\[\s*\{.*?\}\s*\]', raw, re.DOTALL)
        if not m:
            bad += 1; continue
        try:
            qa = json.loads(m.group(0))
        except Exception:
            bad += 1; continue
        for item in qa[:3]:
            q = (item.get('q') or '').strip()
            ans = (item.get('a') or '').strip()
            if not q or not ans: bad += 1; continue
            # verify the cited article number actually matches
            cites = CITE_RE.findall(ans)
            if not cites or a['number'] not in cites:
                bad += 1; continue
            sample = {'messages':[
                {'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':q},
                {'role':'assistant','content':ans},
            ]}
            with out_path.open('a') as f:
                f.write(json.dumps(sample, ensure_ascii=False) + '\n')
            kept += 1
    if (i // BATCH) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed * (len(SAMPLED) - i - BATCH) / max(1, i + BATCH)
        print(f'  {i+BATCH}/{len(SAMPLED)} articles · kept={kept} bad={bad} · {elapsed/60:.1f} min elapsed · ETA {eta/60:.1f} min')
print(f'\n[✓] Tier B done: kept={kept} bad={bad} → {out_path}')

## Cell 5 — Augmentation + combined-question synthesis (regex, free)

In [ ]:
# Paraphrase augmentation via template substitution (free, no LLM)
PARAPHRASE_MAP = {
    'What does Article': ['Explain Article', 'Describe Article', 'Tell me about Article'],
    'Quote Article': ['Give me the verbatim text of Article', 'What is the exact wording of Article'],
    'What is the marginal note': ['What is the heading', 'What is the title'],
    'List all clauses': ['Enumerate the clauses', 'What clauses are in'],
    'Which articles are in': ['What articles are listed under', 'Which Articles fall under'],
}

aug_path = Path(f'{WORK}/dataset_aug.jsonl')
aug_path.write_text('')
with open(f'{WORK}/dataset_tier_a.jsonl') as src:
    for line in src:
        s = json.loads(line)
        q = s['messages'][1]['content']
        for needle, replacements in PARAPHRASE_MAP.items():
            if needle in q:
                for rep in replacements:
                    s2 = json.loads(json.dumps(s))
                    s2['messages'][1]['content'] = q.replace(needle, rep)
                    with aug_path.open('a') as f:
                        f.write(json.dumps(s2, ensure_ascii=False) + '\n')
                break
n_aug = sum(1 for _ in open(aug_path))
print(f'[✓] Paraphrase augmentation: {n_aug} new samples')

In [ ]:
# Combined-question synthesis: 'compare Article X with Article Y' for same-chapter pairs
combined_path = Path(f'{WORK}/dataset_combined.jsonl')
combined_path.write_text('')
combined_count = 0

for part in ir['parts']:
    for ch in part.get('chapters', []) or []:
        arts = [a for a in ch.get('articles', []) or [] if a.get('number') and (a.get('marginal_note') or a.get('text'))]
        if len(arts) < 2: continue
        for _ in range(min(5, len(arts))):
            a1, a2 = random.sample(arts, 2)
            n1, n2 = a1['number'], a2['number']
            mn1 = a1.get('marginal_note', '') or ''
            mn2 = a2.get('marginal_note', '') or ''
            q = f'How do Articles {n1} and {n2} relate to each other?'
            ans = (f'Article {n1} concerns {mn1.rstrip(".").lower() or "its subject"}, '
                   f'while Article {n2} concerns {mn2.rstrip(".").lower() or "its subject"}. '
                   f'Both are found in Part {part["number"]}, Chapter {ch["number"]} of the Constitution. '
                   f'[Art {n1}] [Art {n2}]')
            sample = {'messages':[
                {'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':q},
                {'role':'assistant','content':ans},
            ]}
            with combined_path.open('a') as f:
                f.write(json.dumps(sample, ensure_ascii=False) + '\n')
            combined_count += 1
print(f'[✓] Combined-question samples: {combined_count}')

## Cell 6 — Final pack: merge + dedup + 90/10 split

In [ ]:
import hashlib

def load_jsonl(path):
    return [json.loads(l) for l in open(path)] if Path(path).exists() else []

tier_a = load_jsonl(f'{WORK}/dataset_tier_a.jsonl')
tier_b = load_jsonl(f'{WORK}/dataset_tier_b_raw.jsonl')
aug    = load_jsonl(f'{WORK}/dataset_aug.jsonl')
comb   = load_jsonl(f'{WORK}/dataset_combined.jsonl')
print(f'Tier A: {len(tier_a)}  Tier B: {len(tier_b)}  Aug: {len(aug)}  Combined: {len(comb)}')

all_samples = tier_a + tier_b + aug + comb

# Dedup by question hash
seen = set(); deduped = []
for s in all_samples:
    h = hashlib.md5(s['messages'][1]['content'].encode()).hexdigest()
    if h in seen: continue
    seen.add(h); deduped.append(s)
print(f'After dedup: {len(deduped)} (dropped {len(all_samples) - len(deduped)} duplicates)')

# Remove any question whose text appears in val/heldout prompts (data leakage guard)
leak_prompts = set()
for f in ('val_prompts.jsonl','heldout_prompts.jsonl'):
    for r in load_jsonl(f'{WORK}/{f}'):
        leak_prompts.add(r['prompt'].lower().strip())
clean = [s for s in deduped if s['messages'][1]['content'].lower().strip() not in leak_prompts]
print(f'After leak-guard: {len(clean)} (dropped {len(deduped) - len(clean)})')

# 90/10 shuffle split
random.seed(42); random.shuffle(clean)
split = int(len(clean) * 0.9)
train, val_internal = clean[:split], clean[split:]
print(f'\nFinal: train={len(train)}  val={len(val_internal)}')

# Write
with open(f'{WORK}/dataset_v1_train.jsonl','w') as f:
    for s in train: f.write(json.dumps(s, ensure_ascii=False) + '\n')
with open(f'{WORK}/dataset_v1_val.jsonl','w') as f:
    for s in val_internal: f.write(json.dumps(s, ensure_ascii=False) + '\n')
print('[✓] Written dataset_v1_train.jsonl + dataset_v1_val.jsonl')

# Copy to Drive
import shutil
for f in ['dataset_v1_train.jsonl','dataset_v1_val.jsonl','heldout_prompts.jsonl','val_prompts.jsonl']:
    src = f'{WORK}/{f}'
    if Path(src).exists():
        shutil.copy(src, f'{DRIVE_DIR}/{f}')
        print(f'  → Drive: {f}')
print('\n[✓] All datasets staged in', DRIVE_DIR)

## Cell 7 — Sanity check 5 random training samples

In [ ]:
for i, s in enumerate(random.sample(train, 5)):
    print(f'--- sample {i} ---')
    print('USER:', s['messages'][1]['content'][:160])
    print('ASST:', s['messages'][2]['content'][:240])
    print()

## Done — proceed to `colab_train.ipynb` (Step 3)